In [0]:
print("SmartSense IoT - Databricks environment ready")

In [0]:
print("Spark version:", spark.version)

try:
    scopes = dbutils.secrets.listScopes()
    print("Secret scope support is available.")
    print("Existing scope names:", [s.name for s in scopes])

except Exception as e:
    print("Secret scope access failed.")
    print("Error type:", type(e).__name__)
    print("Error:", str(e)[:500])

In [0]:
scopes = dbutils.secrets.listScopes()

for scope in scopes:
    print(scope.name)

In [0]:
dbutils.secrets.listScopes()

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Add your secret to the scope
w.secrets.put_secret(
    scope="smartsense-secrets",
    key="kafka_bootstrap_server",
    string_value="pkc-921jm.us-east-2.aws.confluent.cloud:9092"
)

In [0]:
w.secrets.put_secret(
    scope="smartsense-secrets",
    key="kafka_api_key",
    string_value="MMXIYRS6NRWMKWSS"
)

In [0]:
w.secrets.put_secret(
    scope="smartsense-secrets",
    key="kafka_api_secret",
    string_value="cflt56ua2hDCP8ynb3S5Hb2yWdNvxnND13Uikh4u94CdpPDNsvX7bUQzlwjJfUOw"
)

In [0]:
BOOTSTRAP_SERVER = dbutils.secrets.get(
    scope="smartsense-secrets",
    key="kafka_bootstrap_server"
)

API_KEY = dbutils.secrets.get(
    scope="smartsense-secrets",
    key="kafka_api_key"
)

API_SECRET = dbutils.secrets.get(
    scope="smartsense-secrets",
    key="kafka_api_secret"
)

print("Confluent credentials loaded successfully.")

In [0]:
KAFKA_TOPIC = "theromstat_readings"

kafka_options = {
    "kafka.bootstrap.servers": BOOTSTRAP_SERVER,
    "subscribe": KAFKA_TOPIC,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="{API_KEY}" '
        f'password="{API_SECRET}";'
    ),
    "startingOffsets": "earliest"
}

print("Kafka configuration prepared.")

In [0]:
try:
    spark.read.format("kafka")
    print("Spark recognizes the Kafka data source.")

except Exception as e:
    print("Kafka data source check failed:")
    print(type(e).__name__, str(e)[:500])

In [0]:
try:
    kafka_df = (
        spark.read
        .format("kafka")
        .options(**kafka_options)
        .option("startingOffsets", "earliest")
        .option("endingOffsets", "latest")
        .load()
    )
    print("Kafka DataFrame created successfully.")
    kafka_df.printSchema()
except Exception as e:
    print("Kafka read failed:")
    print(type(e).__name__, str(e)[:1000])

In [0]:
from pyspark.sql import functions as F

KAFKA_TOPIC = "theromstat_readings"

kafka_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option(
        "kafka.sasl.jaas.config",
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="{API_KEY}" password="{API_SECRET}";'
    )
    .load()
)

print("Streaming DataFrame created.")
kafka_stream.printSchema()

In [0]:
decoded_stream = (
    kafka_stream
    .select(
        F.col("key").cast("string").alias("machine_key"),
        F.col("value").cast("string").alias("raw_json"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp").alias("kafka_timestamp")
    )
)
checkpoint_path = "/Volumes/workspace/default/my_volume/checkpoints/decoded_stream"
dbutils.fs.rm(checkpoint_path, recurse=True)
display(
    decoded_stream,
    checkpointLocation=checkpoint_path,
    outputMode="append"
)

In [0]:
from pyspark.sql import functions as F

bronze_stream = decoded_stream.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)

checkpoint_path = "/Volumes/workspace/default/my_volume/checkpoints/smartsense_bronze"

# Wipe the old checkpoint so Spark reads from 'earliest'
dbutils.fs.rm(checkpoint_path, recurse=True)

bronze_query = (
    bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .outputMode("append")
    .toTable("workspace.default.smartsense_bronze")
)

# Wait for the batch to finish writing to Delta
bronze_query.awaitTermination()

print("Bronze streaming batch completed.")
if bronze_query.lastProgress:
    print("Rows processed:", bronze_query.lastProgress["numInputRows"])

In [0]:
for query in spark.streams.active:
    print("Query name:", query.name)
    print("Query ID:", query.id)
    print("Is active:", query.isActive)
    print("Status:", query.status)
    print("Recent progress:", query.lastProgress)

In [0]:
display(spark.read.table("workspace.default.smartsense_bronze"))